# Experimental Results

## CIFAR10

### Centralized
All 50K data samples are available for the trainer
| eps  | SGDDP  | GEP  |   
|-|-|-|
| $\infty$  | ??? ± ???  | ??? ± ???  |   
|  8 | 47.69333 ± 0.48 | 32.66  ± 1.29|   
|  3 |  ??? ± ??? | 31.86 ± 2.03 |   
|  1 | 38.4666 ± 0.619 | 27.46 ± 3.75  |


### Federated
The data samples are partitioned between **500 clients**. \
There are 10 public clients that approved their data be exposed so we use their gradients to compute the gradient subspace basis. \
**At each step 50 clients are sampled**, preform local train and supply their gradients. \
Gradients are clipped and noised to achieve client level differential privacy according to the requested budget ($\epsilon$).

**Note:** The infinite epsilon case is a sanity check where the SGDDP or GEP is preformed and zero noise is added to the gradients or the ebbeded gradients
#### 10 Classes Each Client
The data samples are i.i.d partitioned between 500 clients. 
| eps  | SGD-DP      | GEP  |
|:-|:-------:|:------:|
|  $\infty$ | 37.14 ± 0.5  |  31.03 ± 0.98 |
|  8 | 23.78 ± 1.6  | 25.6 ± 2.6   |
|  3 | 17.4 ± 1.2 | 23.55 ± 2.7  |
|  1 | 12.4 ± 0.7 | 23.22 ± 1.9 |

#### 2 Classes Each Client
Each client has samples from 2 out of the 10 classes.
| eps  | SGDDP  | GEP  |   
|:---|:---:|:---:|
|$\infty$| 29.2 ± 0.8|16.96  ± 4.96|   
|  8 | 12.49 ± 1.07  |  17.61 ± 3.21 |
|  1 |  12.29 ± 1.49 | 16.5 ± 4.32  |



## putEMG

### Centralized

All data samples gathhered for all 44 clients is available to the trainer as a unified dataset

| eps  |   SGDDP    |     GEP     |
|:---|:----------:|:-----------:|
|$\infty$| |
|  8 | ??? ± ???  |  ??? ± ???  |
|  3 | ??? ± ???  |  ??? ± ???  |
|  1 | ??? ± ???  |  ??? ± ???  |


### Federated

43 people participated in the experiment. Each tested twice. \
Each day is considered as a separate client so we have **86 clients**. \

**5 clients** out of the 86 participants are defined as **public**. \
Clients do not expose their data to the trainer. At each step, **20 clients are sampled**. \
Local train and, DP algorithm and the infinity epsilon sanity check is preformed the same. \

| eps      |   SGDDP    |     GEP     |
|:---------|:----------:|:-----------:|
| $\infty$ | 83.8 ± 0.8 | 74.27 ± 0.5 |
| 8        | 58.3 ± 2.6 | 74.2 ± 0.8  |
| 2        | 52.7 ± 4.5 | 73.3 ± 1.37 |



## CIFAR100

### Centralized

| eps  |   SGDDP   |    GEP    |
|:---|:---------:|:---------:|
|$\infty$|   30.8    |    5.1    |
|  8 | ??? ± ??? | ??? ± ??? |
|  3 | ??? ± ??? | ??? ± ??? |
|  1 | ??? ± ??? | ??? ± ??? |


### Federated


#### 20 Classes Each Client
| eps  | SGDDP  | GEP  |   
|:---|:---:|:---:|
|$\infty$|   |   |   
|  8 | ??? ± ???  |  ??? ± ??? |   
|  3 | ??? ± ???  | ??? ± ???  |   
|  1 |  ??? ± ??? | ??? ± ???  |  

In [1]:
import pandas as pd
from matplotlib import pyplot as plt
from pathlib import Path
# import ipywidgets as widgets

In [17]:
import re
from typing import Optional, Dict, Union

def parse_run_name(name: str) -> Dict[str, Optional[Union[int, float, str]]]:
    """
    Parse run name patterns like:
      - 'eps1_epoch15_gep_seed42'
      - 'epoch20_sgddp_eps0.5_seed7'
      - 'sgddp_epoch3' (some fields may be missing)
    
    Returns a dict with keys: eps (float or int), epoch (int), seed (int), method ('gep' or 'sgd_dp')
    Missing fields are returned as None.
    """
    # Use non-capturing groups and named groups for flexibility
    # Accept eps as int or float: eps1, eps0.5, eps1.0e-3 etc.
    patterns = {
        'eps':   r'eps(?P<eps>[-+]?\d+(?:\.\d+)?(?:e[-+]?\d+)?)',
        'epoch': r'epoch(?P<epoch>\d+)',
        'seed':  r'seed(?P<seed>\d+)',
        'method': r'(?P<method>\bgep\b|\bsgd_dp\b)',
    }
    
    out: Dict[str, Optional[Union[int, float, str]]] = {
        'name': name, 'eps': None, 'epoch': None, 'seed': None, 'method': None
    }

    # Find eps
    m = re.search(patterns['eps'], name, flags=re.IGNORECASE)
    if m:
        eps_str = m.group('eps')
        # cast to float if it looks like a float/scientific, else int
        try:
            out['eps'] = float(eps_str) if any(c in eps_str for c in '.eE') else int(eps_str)
        except ValueError:
            # fallback to float; or leave as None if desired
            try:
                out['eps'] = float(eps_str)
            except ValueError:
                out['eps'] = None

    # Find epoch
    m = re.search(patterns['epoch'], name, flags=re.IGNORECASE)
    if m:
        out['epoch'] = int(m.group('epoch'))

    # Find seed
    m = re.search(patterns['seed'], name, flags=re.IGNORECASE)
    if m:
        out['seed'] = int(m.group('seed'))

    # Find method
    if 'gep' in name:
        out['method'] = 'GEP'
    
    if 'sgd_dp' in name:
        out['method'] = 'SGD_DP'
#     m = re.search(patterns['method'], name, flags=re.IGNORECASE)
#     if m:
#         out['method'] = m.group('method').lower()

    return out

In [22]:
# csv_folder = Path('../csv').resolve()
# assert csv_folder.exists(), f'csv folder does not exist: {csv_folder}'
# csv_files = csv_folder.glob('**/emg_*.csv')
# csv_files = list(csv_files)
# csv_files
csv_folder = Path('C:\\Users\\mb80792\\Downloads\\FEB26\\FEB26')
# csv_folder.exists(),  sum([f.suffix=='.csv' for f in csv_folder.iterdir()])
print('\n'.join([f.name for f in csv_folder.iterdir()]))
f = next(csv_folder.iterdir())
parse_run_name(f.name)
# Path('/home/user1/Downloads/wandb_export_nonoise_cifar10_sgd_dp.csv'),
#  Path('/home/user1/Downloads/eps8_epoch15_sgd_dp.csv'),
# csv_files = [ Path('/home/user1/Downloads/wandb_export_gep_eps8_epochs15.csv')]
# csv_files = [ Path('/home/user1/Downloads/eps8_epoch30_gep_public.csv')]
# csv_files, [f.exists() for f in csv_files] 

eps1_epoch15_gep_seed42.csv
eps1_epoch15_sgd_dp_seed42.csv
eps1_epoch15_sgd_dp_seed42_clsusr10.csv
eps1_epoch15_sgd_dp_seed43_clsusr10.csv
eps1_epoch15_sgd_dp_seed44_clsusr10.csv
eps1_epoch50_seed42_cent_sgddp.csv
eps1_epoch50_seed43_cent_sgddp.csv
eps1_epoch50_seed44_cent_sgddp.csv
eps8_epoch15_gep_seed42.csv
eps8_epoch15_sgd_dp_seed42.csv
eps8_epoch30_gep_public.csv
eps8_epoch30_sgd_dp_seed42.csv
eps8_epoch30_sgd_dp_seed43.csv
eps8_epoch30_sgd_dp_seed44.csv
eps8_epoch50_seed43_cent_sgddp.csv
eps8_epoch50_seed44_cent_sgddp.csv
eps8_epoch50_seed45_cent_sgddp.csv
eps_1_3_8_epoch50_seed42to45_cent_gep.csv


{'name': 'eps1_epoch15_gep_seed42.csv',
 'eps': 1,
 'epoch': 15,
 'seed': 42,
 'method': 'GEP'}

## GEP Centralized

In [24]:
f = csv_folder / 'eps_1_3_8_epoch50_seed42to45_cent_gep.csv'
df = pd.read_csv(f)
df

,Name,batchsize,clip_strategy,clip_value,eps,lr,seed,test_acc
0,batchsize_256_clip_strategy_value_clip_value_3...,256,value,34.5,3,0.0001,45,31.21
1,batchsize_256_clip_strategy_value_clip_value_3...,256,value,34.5,3,0.0001,44,31.42
2,batchsize_256_clip_strategy_value_clip_value_3...,256,value,34.5,3,0.0001,43,30.05
3,batchsize_256_clip_strategy_value_clip_value_3...,256,value,34.5,3,0.0001,42,34.77
4,batchsize_256_clip_strategy_value_clip_value_3...,256,value,34.5,8,0.0001,45,34.46
5,batchsize_256_clip_strategy_value_clip_value_3...,256,value,34.5,8,0.0001,44,32.45
6,batchsize_256_clip_strategy_value_clip_value_3...,256,value,34.5,8,0.0001,43,31.37
7,batchsize_256_clip_strategy_value_clip_value_3...,256,value,34.5,8,0.0001,42,32.39
8,batchsize_256_clip_strategy_value_clip_value_3...,256,value,34.5,1,0.0001,45,23.51
9,batchsize_256_clip_strategy_value_clip_value_3...,256,value,34.5,1,0.0001,44,31.00


In [31]:
for eps in [8,3,1]:
    df_eps = df[df.eps == eps]
    best = df_eps.groupby(['seed'])['test_acc'].max()
    print(f'eps {eps} mean {best.mean()}, std {best.std()}')

eps 8 mean 32.667500000000004, std 1.293686592649085
eps 3 mean 31.862500000000004, std 2.0298008933554716
eps 1 mean 27.465, std 3.7560839536233295


## SGD_DP Centralized

In [37]:
eps1_sgd_dp_cent_files = [f for f in csv_folder.iterdir() if ('cent_sgddp' in f.name and 'eps1' in f.name) ]
eps8_sgd_dp_cent_files = [f for f in csv_folder.iterdir() if ('cent_sgddp' in f.name and 'eps8' in f.name) ]
df1 = pd.concat([pd.read_csv(f) for f in eps1_sgd_dp_cent_files], ignore_index=True)
df8 = pd.concat([pd.read_csv(f) for f in eps8_sgd_dp_cent_files], ignore_index=True)
df8

,Name,batchsize,clip_strategy,clip_value,lr,seed,test_acc
0,batchsize_256_clip_strategy_value_clip_value_0...,256,value,0.131334,0.011744,43,46.01
1,batchsize_256_clip_strategy_value_clip_value_0...,256,value,0.290692,0.013303,43,47.02
2,batchsize_256_clip_strategy_value_clip_value_0...,256,value,0.579774,0.012354,43,47.21
3,batchsize_256_clip_strategy_value_clip_value_0...,256,value,0.546971,0.012435,43,47.21
4,batchsize_512_clip_strategy_value_clip_value_0...,512,value,0.255336,0.020655,44,47.47
5,batchsize_512_clip_strategy_value_clip_value_0...,512,value,0.644441,0.013565,44,48.17
6,batchsize_256_clip_strategy_value_clip_value_0...,256,value,0.917546,0.016767,44,45.58
7,batchsize_256_clip_strategy_value_clip_value_0...,256,value,0.274248,0.021454,44,44.72
8,batchsize_256_clip_strategy_value_clip_value_0...,256,value,0.110766,0.026149,44,41.93
9,batchsize_512_clip_strategy_value_clip_value_0...,512,value,0.391850,0.029543,44,48.17


In [38]:
for eps, df_eps in zip([8,1],[df8, df1]):
    
    best = df_eps.groupby(['seed'])['test_acc'].max()
    print(f'eps {eps} mean {best.mean()}, std {best.std()}')

eps 8 mean 47.69333333333333, std 0.4800347209664461
eps 1 mean 38.46666666666667, std 0.6190584248141152


## SGD_DP Federated

### Classes per client = 2

In [41]:
eps1_sgd_dp_cent_files = [f for f in csv_folder.iterdir() if ('sgd_dp' in f.name and 'cent' not in f.name and 'clsusr10' not in f.name and 'eps1' in f.name) ]
eps8_sgd_dp_cent_files = [f for f in csv_folder.iterdir() if ('sgd_dp' in f.name and 'cent' not in f.name and 'clsusr10' not in f.name and 'eps8' in f.name) ]
df1 = pd.concat([pd.read_csv(f) for f in eps1_sgd_dp_cent_files], ignore_index=True)
df8 = pd.concat([pd.read_csv(f) for f in eps8_sgd_dp_cent_files], ignore_index=True)
df8

,Name,clip,eps,global_lr,lr,lr_dec_rate,n_epochs,seed,wd,val_avg_acc,test_acc
0,clip_0.04629984357537632_eps_8_global_lr_0.810...,0.046300,8,0.810949,0.024159,1.00,15,42,0.001,0.218515,NaN
1,clip_0.010518824743785545_eps_8_global_lr_0.79...,0.010519,8,0.794109,0.065143,0.95,15,42,0.001,0.094033,NaN
2,clip_0.46184289470178824_eps_8_global_lr_0.870...,0.461843,8,0.870979,0.053011,1.00,15,42,0.001,0.106933,NaN
3,clip_0.19595784986806444_eps_8_global_lr_0.813...,0.195958,8,0.813229,0.039998,1.00,15,42,0.001,0.151815,NaN
4,clip_0.9168830314406284_eps_8_global_lr_0.1864...,0.916883,8,0.186417,0.050589,0.95,15,42,0.001,0.108300,NaN
...,...,...,...,...,...,...,...,...,...,...,...
475,clip_0.040420130367399776_eps_8_global_lr_0.84...,0.040420,8,0.841157,0.022357,1.00,30,44,0.001,0.152627,NaN
476,clip_0.02474292649248592_eps_8_global_lr_0.928...,0.024743,8,0.928013,0.020976,0.95,30,44,0.001,0.209745,0.217016
477,clip_0.03565942566228007_eps_8_global_lr_0.815...,0.035659,8,0.815341,0.023590,0.95,30,44,0.001,0.196398,NaN
478,clip_0.02086966021320415_eps_8_global_lr_0.940...,0.020870,8,0.940010,0.020415,0.95,30,44,0.001,0.127306,NaN


In [43]:
for eps, df_eps in zip([8,1],[df8, df1]):
    
    best = df_eps.groupby(['seed'])['val_avg_acc'].max()
    print(f'eps {eps} mean {best.mean()}, std {best.std()}')

eps 8 mean 0.21718690158281376, std 0.01652506895417656
eps 1 mean 0.11469681085433855, std nan


### Classes per client = 10

In [45]:
eps1_sgd_dp_cent_files = [f for f in csv_folder.iterdir() if ('sgd_dp' in f.name and 'cent' not in f.name and 'clsusr10' in f.name and 'eps1' in f.name) ]
# eps8_sgd_dp_cent_files = [f for f in csv_folder.iterdir() if ('sgd_dp' in f.name and 'cent' not in f.name and 'clsusr10' in f.name and 'eps8' in f.name) ]
df1 = pd.concat([pd.read_csv(f) for f in eps1_sgd_dp_cent_files], ignore_index=True)
# df8 = pd.concat([pd.read_csv(f) for f in eps8_sgd_dp_cent_files], ignore_index=True)
df1

,Name,clip,eps,global_lr,lr,lr_dec_rate,n_epochs,seed,wd,val_avg_acc,test_acc
0,clip_0.03826939349263048_eps_1_global_lr_0.249...,0.038269,1,0.249249,0.067285,0.95,15,42,0.001,NaN,NaN
1,clip_0.04412828098236777_eps_1_global_lr_0.452...,0.044128,1,0.452928,0.084211,1.00,15,42,0.001,0.093471,NaN
2,clip_0.03617552285581089_eps_1_global_lr_0.796...,0.036176,1,0.796500,0.097585,0.95,15,42,0.001,0.094125,NaN
3,clip_0.020896685772013132_eps_1_global_lr_0.73...,0.020897,1,0.739766,0.028073,1.00,15,42,0.001,0.093821,NaN
4,clip_0.02267321510150626_eps_1_global_lr_0.564...,0.022673,1,0.564596,0.045588,0.95,15,42,0.001,0.096191,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1003,clip_0.029216557999403623_eps_1_global_lr_0.35...,0.029217,1,0.353446,0.073752,1.00,15,44,0.001,0.099578,0.101795
1004,clip_0.020899864032118062_eps_1_global_lr_0.27...,0.020900,1,0.270528,0.061220,1.00,15,44,0.001,0.106510,0.111933
1005,clip_0.02300100235610992_eps_1_global_lr_0.141...,0.023001,1,0.141981,0.086864,1.00,15,44,0.001,0.100873,0.099466
1006,clip_0.04109787304722066_eps_1_global_lr_0.958...,0.041098,1,0.958907,0.060009,0.95,15,44,0.001,0.099212,NaN


In [47]:
for eps, df_eps in zip([1],[df1]):
    
    best = df_eps.groupby(['seed'])['test_acc'].max()
    print(f'eps {eps} mean {best.mean()}, std {best.std()}')

eps 1 mean 0.12402755434133983, std 0.007293020284924255


In [4]:
csv_files = [ Path('/home/user1/Downloads/eps8_epoch30_sgd_dp_seed42.csv'), Path('/home/user1/Downloads/eps8_epoch30_sgd_dp_seed43.csv'), Path('/home/user1/Downloads/eps8_epoch30_sgd_dp_seed44.csv')]
csv_files, [f.exists() for f in csv_files] 

([PosixPath('/home/user1/Downloads/eps8_epoch30_sgd_dp_seed42.csv'),
  PosixPath('/home/user1/Downloads/eps8_epoch30_sgd_dp_seed43.csv'),
  PosixPath('/home/user1/Downloads/eps8_epoch30_sgd_dp_seed44.csv')],
 [True, True, True])

In [5]:
df_list = [pd.read_csv(f.as_posix()) for f in csv_files]

df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
df

,Name,clip,eps,global_lr,lr,lr_dec_rate,n_epochs,seed,wd,val_avg_acc,test_acc
0,clip_0.04349425299412385_eps_8_global_lr_0.829...,0.043494,8,0.829737,0.019508,0.95,30,42,0.001,0.209748,NaN
1,clip_0.03163805361543916_eps_8_global_lr_0.919...,0.031638,8,0.919386,0.020865,1.00,30,42,0.001,0.135482,NaN
2,clip_0.03338654397101316_eps_8_global_lr_0.939...,0.033387,8,0.939599,0.021542,0.95,30,42,0.001,0.220824,0.239978
3,clip_0.03664337611720077_eps_8_global_lr_0.838...,0.036643,8,0.838273,0.022658,0.95,30,43,0.001,0.130639,NaN
4,clip_0.04322268382551331_eps_8_global_lr_0.880...,0.043223,8,0.880021,0.020846,0.95,30,42,0.001,0.195193,NaN
...,...,...,...,...,...,...,...,...,...,...,...
335,clip_0.040420130367399776_eps_8_global_lr_0.84...,0.040420,8,0.841157,0.022357,1.00,30,44,0.001,0.152627,NaN
336,clip_0.02474292649248592_eps_8_global_lr_0.928...,0.024743,8,0.928013,0.020976,0.95,30,44,0.001,0.209745,0.217016
337,clip_0.03565942566228007_eps_8_global_lr_0.815...,0.035659,8,0.815341,0.023590,0.95,30,44,0.001,0.196398,NaN
338,clip_0.02086966021320415_eps_8_global_lr_0.940...,0.020870,8,0.940010,0.020415,0.95,30,44,0.001,0.127306,NaN


In [7]:
df[df['test_acc'] > 0.19]

,Name,clip,eps,global_lr,lr,lr_dec_rate,n_epochs,seed,wd,val_avg_acc,test_acc
2,clip_0.03338654397101316_eps_8_global_lr_0.939...,0.033387,8,0.939599,0.021542,0.95,30,42,0.001,0.220824,0.239978
9,clip_0.03832166149251896_eps_8_global_lr_0.898...,0.038322,8,0.898277,0.019633,1.00,30,42,0.001,0.222359,0.241069
20,clip_0.041760252295482456_eps_8_global_lr_0.87...,0.041760,8,0.878428,0.018431,1.00,30,42,0.001,0.219275,0.219662
56,clip_0.03320806678603736_eps_8_global_lr_0.909...,0.033208,8,0.909063,0.019542,0.95,30,42,0.001,0.220217,0.226343
80,clip_0.03184441948154111_eps_8_global_lr_0.926...,0.031844,8,0.926814,0.022901,0.95,30,42,0.001,0.224456,0.243796
81,clip_0.03507288204546061_eps_8_global_lr_0.904...,0.035073,8,0.904693,0.023202,0.95,30,42,0.001,0.220384,0.238615
97,clip_0.04296694739521194_eps_8_global_lr_0.810...,0.042967,8,0.810869,0.023362,0.95,30,42,0.001,0.213319,0.224843
138,clip_0.024767484779219424_eps_8_global_lr_0.82...,0.024767,8,0.824210,0.019887,0.95,30,42,0.001,0.236124,0.257431
145,clip_0.033354189944368735_eps_8_global_lr_0.81...,0.033354,8,0.819884,0.020740,0.95,30,43,0.001,0.205692,0.237039
182,clip_0.042456214344730736_eps_8_global_lr_0.91...,0.042456,8,0.916502,0.019613,1.00,30,43,0.001,0.202573,0.231188


In [10]:
best = df.groupby(['seed'])['test_acc'].max()
# best
best.mean(), best.std()

(0.23779708208005648, 0.02023195574338004)

In [6]:
csv_files = [ Path('/home/user1/Downloads/eps1_epoch15_sgd_dp_seed42.csv'), Path('/home/user1/Downloads/eps1_epoch15_gep_seed42.csv')]
csv_files, [f.exists() for f in csv_files] 
df_list = [pd.read_csv(f.as_posix()) for f in csv_files]

# df = pd.concat([pd.read_csv(f) for f in csv_files], ignore_index=True)
# df
for df in df_list:
    best = df.groupby(['seed'])['val_avg_acc'].max()
    print(best)
    # print(best.mean(), best.std())

seed
42    0.114697
Name: val_avg_acc, dtype: float64
seed
42    0.141210
43    0.246276
44    0.206987
Name: val_avg_acc, dtype: float64


In [87]:
res = widgets.RadioButtons(
    options=['SGD_DP', 'GEP_PUBLIC', 'GEP_PRIVATE'],
    description='DP method:',
    disabled=False
)


# Display the radio button widget
display(res)

RadioButtons(description='DP method:', options=('SGD_DP', 'GEP_PUBLIC', 'GEP_PRIVATE'), value='SGD_DP')

In [98]:
good_acc = df[(df['dp_method'] == res.value)]
# good_acc = df[(df['dp_method'] == res.value) & (df['test_avg_acc'] > 0.1)]
good_acc.groupby(['noise-multiplier'])['test_avg_acc'].max()


noise-multiplier
0.000     0.607075
2.016     0.607075
4.720     0.607075
12.790    0.607075
25.000    0.615040
Name: test_avg_acc, dtype: float64

In [99]:
good_acc

,data_name,num-steps,optimizer,lr,num-client-agg,clip,noise-multiplier,seed,history_size,dp_method,epoch_of_best_val,best_val_acc,test_avg_acc
0,putEMG,264,adam,0.010,5,1.00,0.0,42,50,GEP_PUBLIC,34,0.609512,0.607075
1,putEMG,264,adam,0.010,5,0.10,0.0,42,50,GEP_PUBLIC,34,0.609512,0.607075
2,putEMG,264,adam,0.010,5,0.01,0.0,42,50,GEP_PUBLIC,29,0.609512,0.607075
3,putEMG,264,adam,0.001,5,1.00,0.0,42,50,GEP_PUBLIC,29,0.609512,0.607075
4,putEMG,264,adam,0.001,5,0.10,0.0,42,50,GEP_PUBLIC,29,0.609512,0.607075
...,...,...,...,...,...,...,...,...,...,...,...,...,...
57,putEMG,264,adam,0.010,5,0.10,25.0,41,100,GEP_PUBLIC,34,0.609512,0.607075
58,putEMG,264,adam,0.010,5,0.10,25.0,42,100,GEP_PUBLIC,29,0.609512,0.607075
59,putEMG,264,adam,0.001,5,0.10,25.0,40,100,GEP_PUBLIC,29,0.609512,0.607075
60,putEMG,264,adam,0.001,5,0.10,25.0,41,100,GEP_PUBLIC,29,0.609512,0.607075


In [95]:
good_acc.groupby(['seed'])['test_avg_acc'].std()

seed
40    0.002938
41    0.129961
42    0.249766
Name: test_avg_acc, dtype: float64